# PhishGuard — Exploratory Data Analysis

Week 1 deliverable. Covers:
1. Raw dataset inventory and loading
2. Class distribution and source breakdown
3. Body / subject length distributions
4. Missing-value analysis
5. Duplicate detection and deduplication
6. Class-balanced dataset summary

The cleaned dataset is written to `data/processed/cleaned.csv` by the pipeline and used by all downstream notebooks.

In [ ]:
import sys
sys.path.insert(0, "..")   # make src/ importable from notebooks/

import logging
import warnings
warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

# Use a clean, consistent style throughout
plt.rcParams.update({
    "figure.dpi": 110,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 11,
})

from src.data.pipeline import (
    load_nazario,
    load_spamassassin,
    load_enron,
    deduplicate,
    balance_classes,
)

## 1. Load Raw Datasets

Each loader returns a unified DataFrame with columns: `subject`, `body`, `sender`, `label`, `source`.

In [ ]:
df_nazario = load_nazario()
df_spamassassin = load_spamassassin()
df_enron = load_enron()

raw_parts = {
    "Nazario (phishing)": df_nazario,
    "SpamAssassin": df_spamassassin,
    "Enron (ham sample)": df_enron,
}

print("─" * 45)
print(f"{'Dataset':<25} {'Rows':>8} {'Label 0':>8} {'Label 1':>8}")
print("─" * 45)
for name, df in raw_parts.items():
    n0 = (df["label"] == 0).sum()
    n1 = (df["label"] == 1).sum()
    print(f"{name:<25} {len(df):>8,} {n0:>8,} {n1:>8,}")
print("─" * 45)

In [ ]:
# Combine into one raw DataFrame before dedup
df_raw = pd.concat(raw_parts.values(), ignore_index=True)
df_raw["subject"] = df_raw["subject"].fillna("").astype(str)
df_raw["body"]    = df_raw["body"].fillna("").astype(str)
df_raw["sender"]  = df_raw["sender"].fillna("").astype(str)

print(f"Total raw rows : {len(df_raw):,}")
print(f"Phishing (1)   : {(df_raw['label']==1).sum():,}")
print(f"Ham      (0)   : {(df_raw['label']==0).sum():,}")

## 2. Class Distribution & Source Breakdown

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Left: overall class distribution
class_counts = df_raw["label"].value_counts().sort_index()
bars = axes[0].bar(
    ["Ham (0)", "Phishing (1)"],
    class_counts.values,
    color=["#4C9BE8", "#E8644C"],
    width=0.5,
)
for bar in bars:
    axes[0].text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 20,
        f"{bar.get_height():,}",
        ha="center", va="bottom", fontsize=10,
    )
axes[0].set_title("Raw Class Distribution")
axes[0].set_ylabel("Email count")
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))

# Right: emails per source, stacked by class
source_class = (
    df_raw.groupby(["source", "label"])
    .size()
    .unstack(fill_value=0)
    .rename(columns={0: "Ham", 1: "Phishing"})
)
source_class.plot(
    kind="barh", stacked=True,
    ax=axes[1],
    color=["#4C9BE8", "#E8644C"],
)
axes[1].set_title("Emails per Source")
axes[1].set_xlabel("Count")
axes[1].set_ylabel("")
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
axes[1].legend(title="Label")

plt.tight_layout()
plt.show()

## 3. Body & Subject Length Distributions

In [ ]:
df_raw["body_len"]    = df_raw["body"].str.len()
df_raw["subject_len"] = df_raw["subject"].str.len()

ham     = df_raw[df_raw["label"] == 0]
phish   = df_raw[df_raw["label"] == 1]

fig, axes = plt.subplots(2, 2, figsize=(13, 7))
COLORS = {"Ham": "#4C9BE8", "Phishing": "#E8644C"}

for row, (col, title) in enumerate([("body_len", "Body length (chars)"), ("subject_len", "Subject length (chars)")]):
    cap = df_raw[col].quantile(0.97)   # cap outliers at 97th percentile for readability
    bins = 60

    for ax, (label_name, subset) in zip(axes[row], [("Ham", ham), ("Phishing", phish)]):
        data = subset[col].clip(upper=cap)
        ax.hist(data, bins=bins, color=COLORS[label_name], alpha=0.85, edgecolor="none")
        ax.axvline(data.median(), color="black", linewidth=1.2, linestyle="--",
                   label=f"Median: {data.median():.0f}")
        ax.set_title(f"{label_name} — {title}")
        ax.set_xlabel("Characters")
        ax.set_ylabel("Count")
        ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

print("\nBody length summary (chars):")
print(df_raw.groupby("label")["body_len"].describe()[["min","25%","50%","75%","max"]].round(0).to_string())

## 4. Missing Value Analysis

In [ ]:
cols = ["subject", "body", "sender"]

missing = pd.DataFrame({
    "empty_count": [(df_raw[c] == "").sum() for c in cols],
    "empty_pct":   [(df_raw[c] == "").mean() * 100 for c in cols],
}, index=cols)

print("Missing / empty values in raw data:")
print(missing.to_string())

fig, ax = plt.subplots(figsize=(7, 3))
bars = ax.barh(cols, missing["empty_pct"], color="#888", height=0.4)
for bar, pct in zip(bars, missing["empty_pct"]):
    ax.text(bar.get_width() + 0.1, bar.get_y() + bar.get_height() / 2,
            f"{pct:.1f}%", va="center", fontsize=10)
ax.set_xlabel("% empty")
ax.set_title("Empty fields in raw combined dataset")
ax.set_xlim(0, missing["empty_pct"].max() * 1.3 + 1)
plt.tight_layout()
plt.show()

## 5. Deduplication

Duplicates are detected by MD5 hash of whitespace-normalised, lowercased body text. This catches exact reposts and near-identical template emails that only differ in headers.

In [ ]:
before_counts = df_raw["label"].value_counts().sort_index()
df_dedup = deduplicate(df_raw.drop(columns=["body_len", "subject_len"]))
after_counts = df_dedup["label"].value_counts().sort_index()

removed = before_counts - after_counts
print(f"{'':20} {'Before':>8} {'After':>8} {'Removed':>9}")
print("─" * 48)
for label in [0, 1]:
    name = "Ham (0)" if label == 0 else "Phishing (1)"
    print(f"{name:20} {before_counts[label]:>8,} {after_counts[label]:>8,} {removed[label]:>9,}")
print("─" * 48)
print(f"{'Total':20} {before_counts.sum():>8,} {after_counts.sum():>8,} {removed.sum():>9,}")

fig, ax = plt.subplots(figsize=(7, 3.5))
x = [0, 1]
width = 0.35
ax.bar([i - width/2 for i in x], [before_counts[0], before_counts[1]],
       width, label="Before dedup", color=["#4C9BE8", "#E8644C"], alpha=0.5)
ax.bar([i + width/2 for i in x], [after_counts[0], after_counts[1]],
       width, label="After dedup",  color=["#4C9BE8", "#E8644C"], alpha=1.0)
ax.set_xticks(x)
ax.set_xticklabels(["Ham (0)", "Phishing (1)"])
ax.set_ylabel("Count")
ax.set_title("Before vs. After Deduplication")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{int(v):,}"))
ax.legend()
plt.tight_layout()
plt.show()

## 6. Class Balancing & Final Dataset

Undersample the majority class to achieve a 50/50 split. Then save to `data/processed/cleaned.csv`.

In [ ]:
from pathlib import Path
import sys
sys.path.insert(0, "..")

df_balanced = balance_classes(df_dedup)

# Save
out_path = Path("../data/processed/cleaned.csv")
out_path.parent.mkdir(parents=True, exist_ok=True)
df_balanced.to_csv(out_path, index=False)

print(f"Cleaned dataset saved → {out_path.resolve()}")
print(f"\nFinal shape : {df_balanced.shape}")
print(f"Class counts:\n{df_balanced['label'].value_counts().to_string()}")
print(f"\nSource breakdown:\n{df_balanced['source'].value_counts().to_string()}")

In [ ]:
# Final class distribution plot
final_counts = df_balanced["label"].value_counts().sort_index()

fig, ax = plt.subplots(figsize=(6, 3.5))
bars = ax.bar(["Ham (0)", "Phishing (1)"], final_counts.values,
              color=["#4C9BE8", "#E8644C"], width=0.45)
for bar in bars:
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 5,
            f"{bar.get_height():,}", ha="center", va="bottom", fontsize=11)
ax.set_ylabel("Count")
ax.set_title("Final Balanced Dataset — Class Distribution")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{int(v):,}"))
ax.set_ylim(0, final_counts.max() * 1.15)
plt.tight_layout()
plt.show()